In [1]:
from tapas_gmm.dataset.scene import SceneDataset
from pathlib import Path
from tapas_gmm.dataset.bc import BCDataset, BCDataConfig
from torch.utils.data import DataLoader
from tapas_gmm.utils.observation import collate
from tapas_gmm.encoder.encoder import ObservationEncoder, ObservationEncoderConfig
import torch



2026-06-22 23:02:48.642 | INFO     |  Running on cpu


In [2]:
data_root = Path("../outputs/bimanual_dataset")

In [3]:
loaded_dataset = SceneDataset(
    data_root=Path(data_root)
)

2026-06-22 23:02:49.873 | INFO     |  Initializing datasete using ../outputs/bimanual_dataset/metadata.json
2026-06-22 23:02:49.873 | INFO     |  Extracted gt object labels []
2026-06-22 23:02:49.873 | INFO     |  Extracted tsdf object labels []


In [4]:
horizon = 16
n_obs_steps = 2
n_action_steps = 8

bc_config = BCDataConfig(
    fragment_length = horizon + 1,
    cameras=tuple(),
) 

bc_dataset = BCDataset(
    scene_dataset=loaded_dataset,
    config=bc_config,
)

loader = DataLoader(
    bc_dataset,
    collate_fn=collate,
)

2026-06-22 23:02:49.887 | INFO     |  Initializing BCDataset:
2026-06-22 23:02:49.888 | INFO     |    Training on fragments of length 17.
2026-06-22 23:02:49.888 | INFO     |    Loading raw data for encoder.


In [5]:
batch = next(iter(loader))


obs_encoder_config = ObservationEncoderConfig(
    ee_pose=True,
    object_poses=True,
)

obs_encoder = ObservationEncoder(obs_encoder_config)
low_dim_obs, info = obs_encoder.encode(batch)



2026-06-22 23:02:49.932 | INFO     |  No encoder config provided. Using None.
None


In [6]:
policy_obs = low_dim_obs[:, :-1]

robo_state = torch.cat(
    (batch.ee_pose, batch.gripper_state),
    dim=-1,
)

policy_target = robo_state[:, 1:]


In [7]:

print("policy_obs:", policy_obs.shape)
print("robo_state:", robo_state.shape)
print("policy_target:", policy_target.shape)

policy_obs: torch.Size([1, 16, 35])
robo_state: torch.Size([1, 17, 16])
policy_target: torch.Size([1, 16, 16])


In [ ]:
from tapas_gmm.policy.diffusion import (
    DiffusionPolicy,
    DiffusionPolicyConfig,
    DiffusionPolicyTrainingConfig,
)
from tapas_gmm.policy.models.diffusion.conditional_unet1d import ConditionalUnet1DConfig

obs_dim = 35
action_dim = 16

unet_config = ConditionalUnet1DConfig(
    input_dim = action_dim,
    global_cond_dim = obs_dim * n_obs_steps,
)

policy_config = DiffusionPolicyConfig(
    suffix = None,
    obs_as_local_cond=  False,
    obs_as_global_cond = True,
    pred_action_steps_only = False,
    action_dim = action_dim,
    obs_dim = obs_dim,
    horizon = horizon,
    n_obs_steps = n_obs_steps,
    n_action_steps = n_action_steps,
    oa_step_convention = True,
    num_inference_steps = 100,
    training = DiffusionPolicyTrainingConfig(),
    action_scaling = True,
    unet = unet_config,
    obs_encoder = obs_encoder_config,
)

policy = DiffusionPolicy(policy_config)

2026-06-22 23:04:08.341 | INFO     |  Initializing DiffusionPolicy:
2026-06-22 23:04:08.342 | INFO     |    Initializing Policy:
2026-06-22 23:04:09.112 | INFO     |    number of parameters: 66238480
None
UNet parameters: 66238480
action_dim: 16
obs_dim: 35
global_cond_dim: 70
